### Apartado 4: Capa Gold y análisis de negocio

**Objetivo**: tablas agregadas listas para dashboards y Databricks SQL, más optimización Delta.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOG = "aerolinea_trabajo_final"

def guardar_delta(df, schema, tabla):
    nombre = f"{CATALOG}.{schema}.{tabla}"
    (df.write
       .format("delta")
       .mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(nombre))
    print(f"Guardada {nombre}: {spark.table(nombre).count()} filas")

### 4.1 Tablas agregadas Gold

#### a) Retraso promedio por aerolínea y aeropuerto por mes

In [0]:
flights_g = (spark.table(f"{CATALOG}.silver.flights_enriched")
    .withColumn("year", F.year("flight_date"))
    .withColumn("month", F.month("flight_date"))
    .withColumn("year_month", F.date_format("flight_date", "yyyy-MM")))

gold_delay_airline_airport_month = (flights_g
    .groupBy("year_month", "year", "month", "airline_id", "airline_name",
             "origin_airport", "origin_city")
    .agg(F.round(F.avg("dep_delay"), 2).alias("avg_dep_delay"),
         F.round(F.avg("arr_delay"), 2).alias("avg_arr_delay"),
         F.round(F.avg("delay_total"), 2).alias("avg_delay_total"),
         F.round(F.avg(F.col("on_time").cast("int")) * 100, 2).alias("pct_on_time"),
         F.count("*").alias("num_flights"))
    .withColumn("gold_timestamp", F.current_timestamp())
    .orderBy("year_month", "airline_id", "origin_airport"))

display(gold_delay_airline_airport_month)

year_month,year,month,airline_id,airline_name,origin_airport,origin_city,avg_dep_delay,avg_arr_delay,avg_delay_total,pct_on_time,num_flights,gold_timestamp
2025-01,2025,1,AA,American Airlines,ATL,Atlanta,6.9,11.89,18.78,55.7,158,2026-11-05T00:27:51.393Z
2025-01,2025,1,AA,American Airlines,DEN,Denver,6.46,11.68,18.14,54.76,126,2026-11-05T00:27:51.393Z
2025-01,2025,1,AA,American Airlines,DFW,Dallas,7.99,12.74,20.73,59.26,162,2026-11-05T00:27:51.393Z
2025-01,2025,1,AA,American Airlines,JFK,New York,3.96,8.76,12.72,59.57,141,2026-11-05T00:27:51.393Z
2025-01,2025,1,AA,American Airlines,LAS,Las Vegas,6.77,12.8,19.57,52.45,143,2026-11-05T00:27:51.393Z
2025-01,2025,1,AA,American Airlines,LAX,Los Angeles,4.19,9.08,13.27,62.34,154,2026-11-05T00:27:51.393Z
2025-01,2025,1,AA,American Airlines,MIA,Miami,5.01,8.73,13.74,60.62,160,2026-11-05T00:27:51.393Z
2025-01,2025,1,AA,American Airlines,ORD,Chicago,4.76,12.01,16.77,53.42,161,2026-11-05T00:27:51.393Z
2025-01,2025,1,AA,American Airlines,SEA,Seattle,7.6,12.07,19.67,51.45,138,2026-11-05T00:27:51.393Z
2025-01,2025,1,AA,American Airlines,SFO,San Francisco,2.9,8.47,11.37,67.09,158,2026-11-05T00:27:51.393Z


#### b) Ranking de aeropuertos más puntuales

Se ordena por `% de vuelos a tiempo` de los vuelos que salen de cada aeropuerto (desempate: menor retraso promedio). `punctuality_rank = 1` es el más puntual.

In [0]:
w_punct = Window.orderBy(F.col("pct_on_time").desc(), F.col("avg_arr_delay").asc())

gold_airport_punctuality_ranking = (flights_g
    .groupBy("origin_airport", "origin_city", "origin_country")
    .agg(F.count("*").alias("num_departures"),
         F.sum(F.col("on_time").cast("int")).alias("on_time_flights"),
         F.round(F.avg(F.col("on_time").cast("int")) * 100, 2).alias("pct_on_time"),
         F.round(F.avg("dep_delay"), 2).alias("avg_dep_delay"),
         F.round(F.avg("arr_delay"), 2).alias("avg_arr_delay"))
    .withColumn("punctuality_rank", F.dense_rank().over(w_punct))
    .withColumn("gold_timestamp", F.current_timestamp())
    .orderBy("punctuality_rank"))

display(gold_airport_punctuality_ranking)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


origin_airport,origin_city,origin_country,num_departures,on_time_flights,pct_on_time,avg_dep_delay,avg_arr_delay,punctuality_rank,gold_timestamp
ATL,Atlanta,USA,5010,2746,54.81,8.13,13.01,1,2026-11-05T00:27:58.742Z
DEN,Denver,USA,5023,2737,54.49,8.07,12.89,2,2026-11-05T00:27:58.742Z
SFO,San Francisco,USA,4929,2682,54.41,8.39,13.38,3,2026-11-05T00:27:58.742Z
DFW,Dallas,USA,4915,2662,54.16,8.85,13.78,4,2026-11-05T00:27:58.742Z
LAX,Los Angeles,USA,4965,2686,54.1,8.38,13.37,5,2026-11-05T00:27:58.742Z
JFK,New York,USA,4899,2648,54.05,8.11,13.23,6,2026-11-05T00:27:58.742Z
SEA,Seattle,USA,4891,2642,54.02,8.33,13.43,7,2026-11-05T00:27:58.742Z
MIA,Miami,USA,4878,2623,53.77,8.3,13.14,8,2026-11-05T00:27:58.742Z
ORD,Chicago,USA,5007,2686,53.64,8.52,13.51,9,2026-11-05T00:27:58.742Z
LAS,Las Vegas,USA,4962,2642,53.24,8.82,13.91,10,2026-11-05T00:27:58.742Z


#### c) KPI de porcentaje de vuelos a tiempo por aerolínea

Incluye la diferencia frente al promedio general de la red (`pp_vs_network`, en puntos porcentuales).

In [0]:
pct_red = flights_g.agg(F.avg(F.col("on_time").cast("int")) * 100).first()[0]
w_kpi = Window.orderBy(F.col("pct_on_time").desc())

gold_airline_ontime_kpi = (flights_g
    .groupBy("airline_id", "airline_name")
    .agg(F.count("*").alias("num_flights"),
         F.sum(F.col("on_time").cast("int")).alias("on_time_flights"),
         F.round(F.avg(F.col("on_time").cast("int")) * 100, 2).alias("pct_on_time"),
         F.round(F.avg("arr_delay"), 2).alias("avg_arr_delay"))
    .withColumn("network_pct_on_time", F.round(F.lit(pct_red), 2))
    .withColumn("pp_vs_network", F.round(F.col("pct_on_time") - F.lit(pct_red), 2))
    .withColumn("kpi_rank", F.dense_rank().over(w_kpi))
    .withColumn("gold_timestamp", F.current_timestamp())
    .orderBy("kpi_rank"))

display(gold_airline_ontime_kpi)

airline_id,airline_name,num_flights,on_time_flights,pct_on_time,avg_arr_delay,network_pct_on_time,pp_vs_network,kpi_rank,gold_timestamp
UA,United Airlines,8075,4868,60.28,10.32,54.07,6.21,1,2026-11-05T00:28:01.194Z
DL,Delta Air Lines,8292,4991,60.19,10.11,54.07,6.12,2,2026-11-05T00:28:01.194Z
AA,American Airlines,8431,5054,59.95,9.86,54.07,5.88,3,2026-11-05T00:28:01.194Z
WN,Southwest Airlines,8248,4871,59.06,10.23,54.07,4.99,4,2026-11-05T00:28:01.194Z
NK,Spirit Airlines,8206,3494,42.58,19.9,54.07,-11.49,5,2026-11-05T00:28:01.194Z
B6,JetBlue Airways,8227,3476,42.25,19.85,54.07,-11.82,6,2026-11-05T00:28:01.194Z


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


#### d) Tabla de vuelos para el modelo de Machine Learning (insumo del Apartado 6)

Nivel vuelo, con variables de calendario. Se excluyen los vuelos con retrasos imputados para no entrenar con datos inventados.

In [0]:
gold_flights_ml_features = (flights_g
    .filter(~F.col("delay_imputed"))
    .select("flight_id", "flight_date", "year_month", "month",
            F.dayofweek("flight_date").alias("day_of_week"),
            F.quarter("flight_date").alias("quarter"),
            "airline_id", "origin_airport", "destination_airport",
            "dep_delay", "distance_km", "arr_delay")
    .withColumn("gold_timestamp", F.current_timestamp()))

print("Filas para ML:", gold_flights_ml_features.count())
display(gold_flights_ml_features.limit(10))

Filas para ML: 49006


flight_id,flight_date,year_month,month,day_of_week,quarter,airline_id,origin_airport,destination_airport,dep_delay,distance_km,arr_delay,gold_timestamp
FL-100022,2025-01-31,2025-01,1,6,1,WN,SEA,DEN,-25,3971.85,-27,2026-11-05T00:28:02.879Z
FL-100039,2025-02-10,2025-02,2,2,1,DL,SFO,DFW,7,3807.7,-8,2026-11-05T00:28:02.879Z
FL-100068,2025-01-17,2025-01,1,6,1,NK,ORD,DFW,-1,732.25,1,2026-11-05T00:28:02.879Z
FL-100072,2025-04-30,2025-04,4,4,2,NK,MIA,ORD,20,2917.73,33,2026-11-05T00:28:02.879Z
FL-100073,2025-01-22,2025-01,1,4,1,B6,LAS,JFK,-10,651.78,-18,2026-11-05T00:28:02.879Z
FL-100097,2025-05-14,2025-05,5,4,2,B6,LAS,ATL,18,4200.38,27,2026-11-05T00:28:02.879Z
FL-100115,2025-02-04,2025-02,2,3,1,DL,LAS,SEA,-10,3056.14,-2,2026-11-05T00:28:02.879Z
FL-100118,2025-05-27,2025-05,5,3,2,UA,ATL,DFW,-36,2418.84,-29,2026-11-05T00:28:02.879Z
FL-100132,2025-06-05,2025-06,6,5,2,UA,JFK,LAX,0,1837.87,-8,2026-11-05T00:28:02.879Z
FL-100147,2025-02-23,2025-02,2,1,1,DL,LAX,LAS,12,1293.91,10,2026-11-05T00:28:02.879Z


In [0]:
guardar_delta(gold_delay_airline_airport_month, "gold", "delay_by_airline_airport_month")
guardar_delta(gold_airport_punctuality_ranking, "gold", "airport_punctuality_ranking")
guardar_delta(gold_airline_ontime_kpi, "gold", "airline_ontime_kpi")
guardar_delta(gold_flights_ml_features, "gold", "flights_ml_features")

Guardada aerolinea_trabajo_final.gold.delay_by_airline_airport_month: 360 filas


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Guardada aerolinea_trabajo_final.gold.airport_punctuality_ranking: 10 filas
Guardada aerolinea_trabajo_final.gold.airline_ontime_kpi: 6 filas
Guardada aerolinea_trabajo_final.gold.flights_ml_features: 49006 filas


### 4.2 Optimización Delta

- `OPTIMIZE ... ZORDER BY` compacta archivos pequeños y ordena los datos por las columnas más filtradas, para que las consultas lean menos archivos.
- `ANALYZE TABLE ... COMPUTE STATISTICS` actualiza estadísticas para el optimizador de consultas.
- `DESCRIBE HISTORY` muestra el registro de versiones (time travel) de cada tabla.

In [0]:
optimizaciones = {
    f"{CATALOG}.silver.flights_enriched": "flight_date, airline_id",
    f"{CATALOG}.gold.delay_by_airline_airport_month": "airline_id, origin_airport",
    f"{CATALOG}.gold.flights_ml_features": "airline_id, flight_date",
}
for tabla, columnas in optimizaciones.items():
    display(spark.sql(f"OPTIMIZE {tabla} ZORDER BY ({columnas})"))
    spark.sql(f"ANALYZE TABLE {tabla} COMPUTE STATISTICS FOR ALL COLUMNS")
    print(f"Optimizada: {tabla}")

for tabla in [f"{CATALOG}.gold.airport_punctuality_ranking", f"{CATALOG}.gold.airline_ontime_kpi"]:
    spark.sql(f"OPTIMIZE {tabla}")
    print(f"Optimizada: {tabla}")

path,metrics
,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, List(minCubeSize(107374182400), List(0, 0), List(1, 621366), 0, List(0, 0), 0, null), null, 0, 0, 1, 1, false, 0, 0, 1793838496264, 1793838497054, 8, 0, null, List(0, 0), null, 19, 19, 0, 0, null, null, 0)"


Optimizada: aerolinea_trabajo_final.silver.flights_enriched


path,metrics
,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, List(minCubeSize(107374182400), List(0, 0), List(1, 9439), 0, List(0, 0), 0, null), null, 0, 0, 1, 1, false, 0, 0, 1793838500458, 1793838500897, 8, 0, null, List(0, 0), null, 13, 13, 0, 0, null, null, 0)"


Optimizada: aerolinea_trabajo_final.gold.delay_by_airline_airport_month


path,metrics
,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, List(minCubeSize(107374182400), List(0, 0), List(1, 464527), 0, List(0, 0), 0, null), null, 0, 0, 1, 1, false, 0, 0, 1793838503288, 1793838503697, 8, 0, null, List(0, 0), null, 13, 13, 0, 0, null, null, 0)"


Optimizada: aerolinea_trabajo_final.gold.flights_ml_features
Optimizada: aerolinea_trabajo_final.gold.airport_punctuality_ranking
Optimizada: aerolinea_trabajo_final.gold.airline_ontime_kpi


In [0]:
display(spark.sql(f"DESCRIBE HISTORY {CATALOG}.gold.delay_by_airline_airport_month"))
display(spark.sql(f"DESCRIBE DETAIL {CATALOG}.gold.delay_by_airline_airport_month"))

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
5,2026-11-05T00:28:07.000Z,72914905396047,andyduqueep@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, canOverwriteSchema -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""delta.checkpointPolicy"":""v2"",""io.unitycatalog.tableId"":""873b916d-dcc6-4f20-8b62-8face65b4137"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-ae4e3337-ae0f-4e1f-846c-0c415b1b86cc"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-726b67c5-9cd2-41b4-b6c6-3b86abbde8b3""}, statsOnLoad -> true, clusterByAuto -> true, clusteringOnWriteStatus -> Reason for skipping: Nonexistent clustering column, generally when replacing a Liquid table with a table without clustering columns)",null,List(2462558112280833),6cf03762-abbd-4d74-82e0-c6d1c8bd14ab,1104-235349-t1vaqchj-v2n,4,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 10332, numDeletionVectorsRemoved -> 0, numOutputRows -> 360, numOutputBytes -> 9439)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
4,2026-11-04T03:49:17.000Z,72914905396047,andyduqueep@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, canOverwriteSchema -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""delta.checkpointPolicy"":""v2"",""io.unitycatalog.tableId"":""873b916d-dcc6-4f20-8b62-8face65b4137"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-ae4e3337-ae0f-4e1f-846c-0c415b1b86cc"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-726b67c5-9cd2-41b4-b6c6-3b86abbde8b3""}, statsOnLoad -> true, clusterByAuto -> true, clusteringOnWriteStatus -> Reason for skipping: Nonexistent clustering column, generally when replacing a Liquid table with a table without clustering columns)","List(418156220924595, New Job 2026-11-03 22:41:26, 415005914710555, 453393809191993, 72914905396047, manual, 7474658025537115)",null,a15c20dd-26ef-4f14-be88-b3f51ff0ff88,1104-034723-ipj3yhae-v2n,3,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 10280, numDeletionVectorsRemoved -> 0, numOutputRows -> 360, numOutputBytes -> 10332)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
3,2026-11-04T03:35:11.000Z,72914905396047,andyduqueep@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, canOverwriteSchema -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""delta.checkpointPolicy"":""v2"",""io.unitycatalog.tableId"":""873b916d-dcc6-4f20-8b62-8face65b4137"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-ae4e3337-ae0f-4e1f-846c-0c415b1b86cc"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-726b67c5-9cd2-41b4-b6c6-3b86abbde8b3""}, statsOnLoad -> true, clusterByAuto -> true, clusteringOnWriteStatus -> Reason for skipping: Nonexistent clustering column, generally when replacing a Liquid table with a table without clustering columns)",nul

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,02cacc8a-d3ba-4489-883a-4ff78ef9cae5,aerolinea_trabajo_final.gold.delay_by_airline_airport_month,null,,2026-11-04T01:31:29.969Z,2026-11-05T00:28:07.000Z,List(),List(),1,9439,"Map(delta.parquet.compression.codec -> zstd, delta.parquet.format.version.afe.internal -> 2.12.0, delta.enableDeletionVectors -> true, delta.parquet.format.version -> 2.12.0, delta.enableRowTracking -> true, delta.checkpointPolicy -> v2, io.unitycatalog.tableId -> 873b916d-dcc6-4f20-8b62-8face65b4137, delta.rowTracking.materializedRowCommitVersionColumnName -> _row-commit-version-col-ae4e3337-ae0f-4e1f-846c-0c415b1b86cc, delta.rowTracking.materializedRowIdColumnName -> _row-id-col-726b67c5-9cd2-41b4-b6c6-3b86abbde8b3)",3,7,"List(appendOnly, clustering, deletionVectors, domainMetadata, invariants, rowTracking, v2Checkpoint)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",true


In [0]:
%sql
-- Consultas de negocio sobre la capa Gold (listas para dashboards en Databricks SQL)
SELECT kpi_rank, airline_name, pct_on_time, pp_vs_network, avg_arr_delay, num_flights
FROM aerolinea_trabajo_final.gold.airline_ontime_kpi
ORDER BY kpi_rank

kpi_rank,airline_name,pct_on_time,pp_vs_network,avg_arr_delay,num_flights
1,United Airlines,60.28,6.21,10.32,8075
2,Delta Air Lines,60.19,6.12,10.11,8292
3,American Airlines,59.95,5.88,9.86,8431
4,Southwest Airlines,59.06,4.99,10.23,8248
5,Spirit Airlines,42.58,-11.49,19.9,8206
6,JetBlue Airways,42.25,-11.82,19.85,8227


In [0]:
%sql
SELECT punctuality_rank, origin_airport, origin_city, pct_on_time, avg_dep_delay
FROM aerolinea_trabajo_final.gold.airport_punctuality_ranking
ORDER BY punctuality_rank

punctuality_rank,origin_airport,origin_city,pct_on_time,avg_dep_delay
1,ATL,Atlanta,54.81,8.13
2,DEN,Denver,54.49,8.07
3,SFO,San Francisco,54.41,8.39
4,DFW,Dallas,54.16,8.85
5,LAX,Los Angeles,54.1,8.38
6,JFK,New York,54.05,8.11
7,SEA,Seattle,54.02,8.33
8,MIA,Miami,53.77,8.3
9,ORD,Chicago,53.64,8.52
10,LAS,Las Vegas,53.24,8.82


In [0]:
%sql
SHOW TABLES IN aerolinea_trabajo_final.gold

database,tableName,isTemporary
gold,airline_ontime_kpi,false
gold,airport_punctuality_ranking,false
gold,delay_by_airline_airport_month,false
gold,flights_ml_features,false
